# Notebook 14 — ML for Clinical Omics

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 14 of 15  
**Time estimate:** 90 minutes

> Clinical ML is where computational biology meets patient outcomes.
> The stakes are different — wrong predictions harm patients — and the constraints
> are different: small n, multi-modal data, batch effects, and regulatory requirements
> for interpretability. Standard ML pipelines must be adapted.

---
## Step 1 — Motivation

TCGA has RNA-seq profiles for ~10,000 tumor samples across 33 cancer types.
Can we predict overall survival from expression? Can we classify tumor-of-origin
from expression alone? This is real-world clinical ML: small n relative to p,
censored survival data, batch effects across institutions, class imbalance,
and the need for interpretable results that a clinician can act on.

---
## Step 2 — Intuition

**Multi-modal omics integration:**
A patient has DNA (mutations, copy number), RNA (expression), protein (RPPA),
clinical (age, stage). Each modality is a different feature space.
Strategies:
- **Early integration:** Concatenate all features → train one model (simple but ignores
  modality structure)
- **Late integration (ensemble):** Train one model per modality, combine predictions
  (handles missing modalities gracefully)
- **Intermediate integration:** Learn modality-specific embeddings, then combine
  (e.g., multi-view PCA, or neural encoder per modality)

**Survival analysis:**
Standard classification assumes we know the outcome. In clinical data, we often
have *censored* observations: the patient survived past the end of the study
(we know they survived at least T months, but not how long).
- Cox proportional hazards model: the standard clinical ML baseline
- Random Survival Forest (RSF): ensemble extension that handles non-linearity

**Batch effect correction:**
Samples from different hospitals or sequencing runs have systematic technical
differences. Combat (empirical Bayes) and harmonization are standard approaches.
Always check PCA before and after correction.

**Transfer learning in clinical ML:**
Pre-trained models from large datasets (e.g., GTEx expression profiles) can be
fine-tuned on small clinical cohorts — reduces overfitting when n is small.

---
## Step 3 — Biological Background

**TCGA (The Cancer Genome Atlas):**
Pan-cancer analysis. 33 cancer types, ~10,000 patients.
Multi-modal: WXS (somatic mutations), SCNA (copy number), RNA-seq (expression),
RPPA (protein), clinical outcomes.
Key paper: The Cancer Genome Atlas Research Network (2013); TCGA PanCanAtlas (2018).

**PAM50 breast cancer subtypes:**
50-gene expression signature that classifies breast cancer into:
LumA (most common, best prognosis), LumB, HER2-enriched, Basal-like (triple-negative),
Normal-like. Validated clinically for treatment decisions.

**Liquid biopsy:**
Circulating tumor DNA (ctDNA) in blood → classify tumor of origin using methylation
signatures. ML (random forest on methylation patterns) enables early cancer detection.
Galleri (Grail) test uses cfDNA methylation + ML.

**Clinical endpoint definitions:**
- **OS (Overall Survival):** time from diagnosis to death
- **PFS (Progression-Free Survival):** time to disease progression or death
- **Response (pCR, ORR):** pathological complete response, objective response rate
- Censoring: patient left study, study ended — we know survival ≥ T, not exact time

**Prognostic vs. predictive biomarkers:**
- **Prognostic:** correlates with outcome regardless of treatment (e.g., stage)
- **Predictive:** identifies patients who will benefit from a specific treatment
  (e.g., HER2 amplification → responds to trastuzumab)

---
## Step 4 — Mathematical Explanation

**Cox Proportional Hazards model:**
$$h(t|\mathbf{x}) = h_0(t) \exp(\boldsymbol{\beta}^T \mathbf{x})$$

- $h(t|\mathbf{x})$: hazard at time $t$ for a patient with features $\mathbf{x}$
- $h_0(t)$: baseline hazard (shared by all patients)
- $\boldsymbol{\beta}$: feature weights (the ML parameters)
- Assumption: the effect of features on hazard is *proportional* (constant ratio
  across time)

**Partial likelihood (Cox):**
$$L(\boldsymbol{\beta}) = \prod_{i: \delta_i=1} \frac{\exp(\boldsymbol{\beta}^T \mathbf{x}_i)}{\sum_{j \in \mathcal{R}(t_i)} \exp(\boldsymbol{\beta}^T \mathbf{x}_j)}$$

where $\mathcal{R}(t_i)$ is the risk set at time $t_i$ (all patients still alive),
$\delta_i = 1$ if death observed (not censored).

**C-index (concordance index):**
Fraction of all patient pairs where the patient who died sooner had the higher
predicted risk. Analogous to AUROC for survival data.
$$\text{C-index} = P(\hat{r}_i > \hat{r}_j \mid t_i < t_j, \delta_i = 1)$$
C = 0.5: random; C = 1: perfect; C = 0: perfectly reversed.

**Kaplan-Meier estimator:**
$$\hat{S}(t) = \prod_{t_i \leq t} \left(1 - \frac{d_i}{n_i}\right)$$

where $d_i$ = deaths at $t_i$, $n_i$ = patients at risk. Step function estimate
of survival probability over time.

In [ ]:
# Step 6 — Python: Clinical omics ML
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

# ---- Simulate TCGA-like multi-modal dataset ----
rng = np.random.default_rng(42)
n_patients = 300
n_cancer_types = 5
CANCER_TYPES = ['BRCA', 'LUAD', 'COAD', 'KIRC', 'GBM']

# RNA-seq expression (log-TPM, 500 genes)
n_genes = 500
gene_means = np.zeros((n_cancer_types, n_genes))
for c in range(n_cancer_types):
    # Each cancer type has a distinct signature in different gene sets
    gene_means[c, c*40:(c+1)*40] = 3.0
    gene_means[c, 200+c*20:200+c*20+10] = -2.0

y_cancer = np.repeat(np.arange(n_cancer_types), n_patients // n_cancer_types)
X_rna = np.vstack([rng.normal(gene_means[c], 0.8, (n_patients//n_cancer_types, n_genes))
                    for c in range(n_cancer_types)])

# Mutation burden (binary features for 20 commonly mutated genes)
mut_probs = np.array([0.3, 0.4, 0.2, 0.1, 0.5] * 4)  # 20 genes
X_mut = np.vstack([rng.binomial(1, mut_probs, (n_patients//n_cancer_types, 20))
                    for _ in range(n_cancer_types)]).astype(float)
# Add cancer-type-specific mutation patterns
for c in range(n_cancer_types):
    mask = y_cancer == c
    X_mut[mask, c*4:(c+1)*4] += rng.binomial(1, 0.6, (mask.sum(), 4))
    X_mut = np.clip(X_mut, 0, 1)

# Clinical features: age, stage (1-4), tumor size
X_clin = np.column_stack([
    rng.normal(60, 10, n_patients),    # age
    rng.randint(1, 5, n_patients),      # stage
    rng.normal(3, 1.5, n_patients),    # tumor size (cm)
])

# Simulated survival (OS in months)
# Higher risk = later stage, higher tumor burden (some cancer-type differences)
baseline_risk = (X_clin[:, 1] - 1) * 0.3 + X_clin[:, 2] * 0.2
survival_months = rng.exponential(scale=np.exp(-baseline_risk) * 30)
censoring_time = rng.uniform(12, 60, n_patients)
os_time = np.minimum(survival_months, censoring_time)
os_event = (survival_months <= censoring_time).astype(int)  # 1=event, 0=censored

print(f'Dataset: {n_patients} patients, {n_cancer_types} cancer types')
print(f'RNA-seq: {X_rna.shape[1]} genes')
print(f'Mutations: {X_mut.shape[1]} genes')
print(f'Clinical: {X_clin.shape[1]} features')
print(f'Events (deaths): {os_event.sum()} ({os_event.mean():.0%})')
print(f'Median OS: {np.median(os_time):.1f} months')

# ---- Task 1: Cancer type classification from RNA-seq ----
le = LabelEncoder()
y_enc = le.fit_transform(y_cancer)

scaler_rna = StandardScaler()
X_rna_std = scaler_rna.fit_transform(X_rna)

pca = PCA(n_components=50)
X_rna_pca = pca.fit_transform(X_rna_std)

print(f'\nPCA top-3 components explain: {pca.explained_variance_ratio_[:3].sum():.1%} variance')

cv = StratifiedKFold(5, shuffle=True, random_state=42)
for name, X_feat, clf in [
    ('RNA PCA (50 PCs) + LR', X_rna_pca,
     LogisticRegression(C=1, max_iter=2000, multi_class='multinomial')),
    ('RNA PCA + RF (100)', X_rna_pca,
     RandomForestClassifier(100, random_state=42)),
    ('Mutations + RF', StandardScaler().fit_transform(X_mut),
     RandomForestClassifier(100, random_state=42)),
    ('Multi-modal concat + RF', np.hstack([X_rna_pca, X_mut, StandardScaler().fit_transform(X_clin)]),
     RandomForestClassifier(100, random_state=42)),
]:
    acc = cross_val_score(clf, X_feat, y_enc, cv=cv, scoring='accuracy')
    print(f'{name}: accuracy={acc.mean():.3f}±{acc.std():.3f}')

# ---- Task 2: Cox PH survival model (manual partial likelihood) ----
def cox_c_index(risk_scores, os_time, os_event):
    """Compute concordance index from risk scores."""
    concordant = 0
    discordant = 0
    tied = 0
    n = len(os_time)
    for i in range(n):
        if os_event[i] == 0:
            continue  # only consider observed deaths
        for j in range(n):
            if os_time[j] <= os_time[i]:
                continue
            if risk_scores[i] > risk_scores[j]:
                concordant += 1
            elif risk_scores[i] < risk_scores[j]:
                discordant += 1
            else:
                tied += 0.5
    total = concordant + discordant + tied
    return (concordant + 0.5 * tied) / max(total, 1)

# Simple Cox-like: Ridge regression on survival time (crude but illustrative)
# In real work: use lifelines.CoxPHFitter or scikit-survival
X_surv = np.column_stack([X_rna_pca[:, :20], X_clin])
scaler_surv = StandardScaler()
X_surv_std = scaler_surv.fit_transform(X_surv)

# Cross-validated C-index (manual)
kf = StratifiedKFold(5, shuffle=True, random_state=42)
c_indices = []
for tr, te in kf.split(X_surv_std, os_event):
    reg = Ridge(alpha=1.0)
    # Use Ridge to predict log survival time (concordance-oriented)
    reg.fit(X_surv_std[tr], -os_time[tr])  # negative: high risk = short survival
    risk_te = reg.predict(X_surv_std[te])
    c = cox_c_index(risk_te, os_time[te], os_event[te])
    c_indices.append(c)
print(f'\nCox-like survival C-index (5-fold): {np.mean(c_indices):.3f}±{np.std(c_indices):.3f}')
print(f'(Baseline C-index for random predictor: ~0.50)')

# ---- Kaplan-Meier from scratch ----
def kaplan_meier(times, events):
    """Compute KM survival curve."""
    order = np.argsort(times)
    times_sorted = times[order]
    events_sorted = events[order]
    unique_times = np.unique(times_sorted[events_sorted == 1])
    n = len(times)
    S = 1.0
    km_times = [0]
    km_surv = [1.0]
    for t in unique_times:
        n_risk = np.sum(times_sorted >= t)
        n_events = np.sum((times_sorted == t) & (events_sorted == 1))
        S *= (1 - n_events / max(n_risk, 1))
        km_times.append(t)
        km_surv.append(S)
    return np.array(km_times), np.array(km_surv)

# KM curves per cancer type
km_curves = {}
for c, cname in enumerate(CANCER_TYPES):
    mask = y_cancer == c
    t, s = kaplan_meier(os_time[mask], os_event[mask])
    km_curves[cname] = (t, s)
print(f'\nKaplan-Meier curves computed for {len(CANCER_TYPES)} cancer types')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
cmap = plt.cm.get_cmap('tab10', n_cancer_types)

# Panel A: PCA colored by cancer type
ax = axes[0]
for c, cname in enumerate(CANCER_TYPES):
    mask = y_cancer == c
    ax.scatter(X_rna_pca[mask, 0], X_rna_pca[mask, 1], c=[cmap(c)],
                 s=15, alpha=0.7, label=cname)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('A. PCA of RNA-seq\ncolored by cancer type')
ax.legend(fontsize=7, loc='upper right')

# Panel B: Kaplan-Meier survival curves
ax = axes[1]
for c, cname in enumerate(CANCER_TYPES):
    t, s = km_curves[cname]
    ax.step(t, s, where='post', color=cmap(c), lw=2, label=cname)
ax.set_xlabel('Time (months)')
ax.set_ylabel('Survival probability')
ax.set_title('B. Kaplan-Meier curves\n(per cancer type)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='grey', ls=':', lw=0.8)

# Panel C: Multi-modal vs. single-modal accuracy
ax = axes[2]
modal_names = ['RNA\n(PCA+LR)', 'RNA\n(PCA+RF)', 'Mutations\n(RF)', 'Multi-modal\n(RF)']
modal_accs = []  # recompute for plot
for X_feat, clf in [
    (X_rna_pca, LogisticRegression(C=1, max_iter=2000, multi_class='multinomial')),
    (X_rna_pca, RandomForestClassifier(100, random_state=42)),
    (StandardScaler().fit_transform(X_mut), RandomForestClassifier(100, random_state=42)),
    (np.hstack([X_rna_pca, X_mut, StandardScaler().fit_transform(X_clin)]),
     RandomForestClassifier(100, random_state=42)),
]:
    a = cross_val_score(clf, X_feat, y_enc, cv=cv, scoring='accuracy')
    modal_accs.append(a)

means = [a.mean() for a in modal_accs]
stds = [a.std() for a in modal_accs]
colors_bar = ['steelblue', 'tomato', 'green', 'orange']
ax.bar(range(4), means, yerr=stds, capsize=4, color=colors_bar, edgecolor='black', alpha=0.8)
ax.set_xticks(range(4))
ax.set_xticklabels(modal_names, fontsize=8)
ax.axhline(1/n_cancer_types, color='grey', ls='--', lw=0.8, label='Random')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('C. Modality comparison\n(multi-modal integration)')
ax.set_ylim(0, 1)
ax.legend(fontsize=7)

# Panel D: Risk stratification (high vs. low predicted risk)
ax = axes[3]
# Split patients into high/low risk by median predicted risk
reg_final = Ridge(alpha=1.0)
reg_final.fit(X_surv_std, -os_time)  # predict risk
risk_pred = reg_final.predict(X_surv_std)
median_risk = np.median(risk_pred)
high_risk = risk_pred >= median_risk

for label, mask, color in [('High risk', high_risk, 'tomato'), ('Low risk', ~high_risk, 'steelblue')]:
    t, s = kaplan_meier(os_time[mask], os_event[mask])
    ax.step(t, s, where='post', color=color, lw=2, label=label)
ax.set_xlabel('Time (months)')
ax.set_ylabel('Survival probability')
ax.set_title('D. Risk stratification\n(high vs. low predicted risk)')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='grey', ls=':', lw=0.8)

plt.suptitle('Module 13 NB14: ML for Clinical Omics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ml_clinical_omics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement the Kaplan-Meier estimator from scratch and verify it matches
   the `lifelines.KaplanMeierFitter` (if available) on the synthetic data.
2. Implement the concordance index from scratch. Show that a random risk predictor
   gives C ≈ 0.5.
3. Simulate a batch effect: add a systematic shift to RNA-seq features for
   patients 0–149 (a different hospital). Show the batch separates in PCA.
   Apply mean-centering per batch and show it is reduced.
4. Implement late integration: train one LR model per modality (RNA, mutations,
   clinical), average the predicted probabilities. Compare to early (concatenation)
   and single-modality approaches.

---
## Step 10 — Quiz

1. What is censoring in survival analysis? Why can't we simply exclude censored patients?
2. Write the Cox PH hazard function. What is the proportional hazards assumption?
3. What does the C-index measure? What values indicate random vs. perfect performance?
4. What is the difference between a prognostic and a predictive biomarker?
5. Describe three challenges specific to clinical ML that don't arise in standard
   benchmarks.

---
## Step 12 — Reflection

> *[In one paragraph: explain why the Cox model is still widely used in clinical ML
> despite its proportional hazards assumption being frequently violated, and
> describe what validation step you would perform to check whether the assumption holds.]*

---
*Next: `15_mini_projects_and_assessment.ipynb`*